In [1]:
# ==============================
# PART 1 — CALCULATION
# Cluster "Description" and correlate with "Type de constatation Texte"
# ==============================

import os
import re
import unicodedata
import pandas as pd
from collections import Counter
from rapidfuzz import fuzz
from rapidfuzz.distance import Levenshtein
from tqdm import tqdm

# ------------------------------
# CONFIGURATION
# ------------------------------
csv_file = "20250903_Extrait_Constatations_F2.csv"
desc_col = "Description"
type_col = "Type de constatation Texte"
sep = ";"

min_occurrences = 7
threshold_similarity = 90
max_typo_chars = 4
checkpoint_every = 50

# ------------------------------
# TEXT CLEANING
# ------------------------------
def normalize_text(s: str) -> str:
    if pd.isna(s) or not str(s).strip():
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKC", s)
    s = s.replace("\xa0", " ").strip()
    return s

def clean_text_block(text: str) -> str:
    if not text:
        return ""
    text = normalize_text(text).lower()
    text = re.sub(r"[^a-zà-öø-ÿœæçß0-9\s-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    words = [w for w in text.split() if len(w) > 1]
    return " ".join(words)

def join_variations_with_counts(variants, counter, delim=" | "):
    return delim.join([f"{v} ({counter.get(v,0)})" for v in variants])

# ------------------------------
# CLUSTERING HELPER
# ------------------------------
def cluster_texts(candidates, counter, label="Text"):
    cluster_groups = {}
    for cluster in tqdm(candidates.keys(), desc=f"Fuzzy clustering {label}"):
        found = False
        for rep in list(cluster_groups.keys()):
            sim = fuzz.ratio(cluster, rep)
            edit_d = Levenshtein.distance(cluster, rep)
            if sim >= threshold_similarity or edit_d <= max_typo_chars:
                cluster_groups[rep].append(cluster)
                found = True
                break
        if not found:
            cluster_groups[cluster] = [cluster]
    return cluster_groups

def summarize_clusters(cluster_groups, counter, total):
    rows = []
    for rep, variants in cluster_groups.items():
        cnt = sum(counter.get(v, 0) for v in variants)
        per_mille = round(cnt / total * 1000, 2) if total > 0 else 0
        variations_text = join_variations_with_counts(variants, counter)
        rows.append({
            "Cluster_Representative": rep,
            "Count": cnt,
            "PerMille": per_mille,
            "Variations": variations_text,
        })
    return pd.DataFrame(rows).sort_values(by="Count", ascending=False)

# ------------------------------
# LOAD DATA
# ------------------------------
print("\n=== Loading data ===")
df = pd.read_csv(csv_file, delimiter=sep, dtype=str)
df.columns = [normalize_text(c) for c in df.columns]

if desc_col not in df.columns or type_col not in df.columns:
    raise KeyError(f"Columns '{desc_col}' or '{type_col}' not found. Available: {df.columns.tolist()}")

df = df.dropna(subset=[desc_col, type_col])
df = df[(df[desc_col].str.strip() != "") & (df[type_col].str.strip() != "")]
df = df.reset_index(drop=True)
print(f"Loaded {len(df)} valid rows.\n")

# ------------------------------
# CLUSTER DESCRIPTIONS
# ------------------------------
print("=== Clustering on Description field ===")
cleaned_desc = [clean_text_block(t) for t in tqdm(df[desc_col])]
cleaned_desc = [t for t in cleaned_desc if t]
desc_counter = Counter(cleaned_desc)
desc_total = sum(desc_counter.values())

desc_candidates = {k:v for k,v in desc_counter.items() if v >= min_occurrences}
print(f"Found {len(desc_candidates)} Description candidates (≥{min_occurrences} occurrences).")

desc_clusters = cluster_texts(desc_candidates, desc_counter, label="Description")
desc_summary_df = summarize_clusters(desc_clusters, desc_counter, desc_total)
desc_summary_df.to_csv("description_clusters_summary.csv", sep=sep, index=False)
print(f"✅ Saved description_clusters_summary.csv with {len(desc_summary_df)} clusters.\n")

# ------------------------------
# CORRELATE Description ↔ Type
# ------------------------------
print("=== Correlating Description clusters ↔ Type de constatation Texte ===")
desc_texts = [clean_text_block(t) for t in df[desc_col]]
type_texts = [normalize_text(t) for t in df[type_col]]

corr_rows = []
for i, (rep, variants) in enumerate(tqdm(desc_clusters.items(), desc="Correlating")):
    matched_rows = [idx for idx, text in enumerate(desc_texts) if text in variants]
    if not matched_rows:
        continue
    matched_types = [type_texts[idx] for idx in matched_rows if type_texts[idx]]
    type_counts = Counter(matched_types)
    total_corr = sum(type_counts.values())
    for t_type, t_cnt in type_counts.items():
        t_per_mille = round(t_cnt / total_corr * 1000, 2) if total_corr > 0 else 0
        corr_rows.append({
            "Description_Cluster": rep,
            "Description_Count": sum(desc_counter.get(v, 0) for v in variants),
            "Description_PerMille": round(sum(desc_counter.get(v, 0) for v in variants) / desc_total * 1000, 2),
            "Type_de_constatation": t_type,
            "Type_Count": t_cnt,
            "Type_PerMille": t_per_mille,
        })
    if (i + 1) % checkpoint_every == 0:
        pd.DataFrame(corr_rows).to_csv(f"partial_type_corr_checkpoint_{i+1}.csv", sep=sep, index=False)
        print(f"Checkpoint {i+1}: {len(corr_rows)} correlation rows saved.")

corr_df = pd.DataFrame(corr_rows)
corr_df.to_csv("description_type_correlations.csv", sep=sep, index=False)
print(f"\n✅ Saved description_type_correlations.csv with {len(corr_df)} rows.")



=== Loading data ===
Loaded 7625 valid rows.

=== Clustering on Description field ===


100%|███████████████████████████████████████████████████████████████████████████| 7625/7625 [00:00<00:00, 40334.27it/s]


Found 70 Description candidates (≥7 occurrences).


Fuzzy clustering Description: 100%|███████████████████████████████████████████████████| 70/70 [00:00<00:00, 209.04it/s]


✅ Saved description_clusters_summary.csv with 66 clusters.

=== Correlating Description clusters ↔ Type de constatation Texte ===


Correlating: 100%|████████████████████████████████████████████████████████████████████| 66/66 [00:00<00:00, 508.64it/s]

Checkpoint 50: 214 correlation rows saved.

✅ Saved description_type_correlations.csv with 277 rows.
